Etapa uno Limpieza de dataset 


In [4]:
import pandas as pd
import numpy as np

# Cargar el dataset
print("Cargando dataset...")
df = pd.read_csv('../DataSet/household_energy_consumption.csv')

print("=" * 70)
print("ANÁLISIS INICIAL DEL DATASET")
print("=" * 70)
print(f"\nDimensiones originales: {df.shape}")
print(f"Columnas: {list(df.columns)}")

# ============================================================================
# 1. VERIFICAR DATOS FALTANTES
# ============================================================================
print("\n" + "=" * 70)
print("1. VERIFICACIÓN DE DATOS FALTANTES")
print("=" * 70)

datos_faltantes = df.isnull().sum()
print(f"\nDatos faltantes por columna:")
print(datos_faltantes)
print(f"\nTotal de filas con datos faltantes: {df.isnull().any(axis=1).sum()}")

if datos_faltantes.sum() == 0:
    print("✓ No hay datos faltantes en el dataset")
else:
    print("⚠ Se encontraron datos faltantes")

# ============================================================================
# 2. ANÁLISIS DE MEDICIONES POR CASA
# ============================================================================
print("\n" + "=" * 70)
print("2. ANÁLISIS DE MEDICIONES POR CASA")
print("=" * 70)

mediciones_por_casa = df.groupby('Household_ID').size()
print(f"\nEstadísticas de mediciones por casa:")
print(mediciones_por_casa.describe())

print(f"\nDistribución de mediciones:")
print(mediciones_por_casa.value_counts().sort_index())

# Identificar casas con más mediciones
casas_con_mas = mediciones_por_casa[mediciones_por_casa > 7]
if len(casas_con_mas) > 0:
    print(f"\nCasas con más de 7 mediciones:")
    for casa, num_mediciones in casas_con_mas.items():
        print(f"  - {casa}: {num_mediciones} mediciones")
else:
    print(f"\n✓ Todas las casas tienen 7 mediciones")

# ============================================================================
# 3. CONVERTIR HAS_AC A VALORES BINARIOS
# ============================================================================
print("\n" + "=" * 70)
print("3. CONVERTIR HAS_AC A VALORES BINARIOS")
print("=" * 70)

print(f"\nValores únicos antes de la conversión:")
print(df['Has_AC'].value_counts())

# Convertir Yes/No a 1/0
df['Has_AC'] = df['Has_AC'].map({'Yes': 1, 'No': 0})

print(f"\nValores únicos después de la conversión:")
print(df['Has_AC'].value_counts().sort_index())
print("✓ Conversión completada: 'Yes' → 1, 'No' → 0")

# ============================================================================
# 4. ELIMINAR SOLO LA FILA EXTRA DE CASAS CON MÁS DE 7 MEDICIONES
# ============================================================================
print("\n" + "=" * 70)
print("4. ELIMINAR SOLO LA FILA EXTRA (MANTENER 7 MEDICIONES POR CASA)")
print("=" * 70)

print(f"\nFilas antes de eliminar: {len(df)}")

# Para cada casa con más de 7 mediciones, eliminar solo las filas extras
df_limpio = df.copy()
casas_procesadas = []

for casa in casas_con_mas.index:
    num_mediciones = mediciones_por_casa[casa]
    filas_a_eliminar = num_mediciones - 7
    
    # Obtener índices de todas las filas de esta casa
    indices_casa = df_limpio[df_limpio['Household_ID'] == casa].index
    
    # Eliminar las últimas filas extras (mantener las primeras 7)
    indices_a_eliminar = indices_casa[-filas_a_eliminar:]
    
    print(f"\nCasa {casa}:")
    print(f"  - Mediciones totales: {num_mediciones}")
    print(f"  - Filas a eliminar: {filas_a_eliminar}")
    print(f"  - Índices eliminados: {list(indices_a_eliminar)}")
    
    df_limpio = df_limpio.drop(indices_a_eliminar)
    casas_procesadas.append(casa)

filas_eliminadas = len(df) - len(df_limpio)
print(f"\n{'='*40}")
print(f"Filas después de eliminar: {len(df_limpio)}")
print(f"Total de filas eliminadas: {filas_eliminadas}")
print(f"Casas procesadas: {casas_procesadas}")

# ============================================================================
# 5. VERIFICACIÓN FINAL
# ============================================================================
print("\n" + "=" * 70)
print("5. VERIFICACIÓN FINAL DEL DATASET LIMPIO")
print("=" * 70)

print(f"\nDimensiones finales: {df_limpio.shape}")
print(f"Total de casas: {df_limpio['Household_ID'].nunique()}")

# Verificar que todas las casas tengan exactamente 7 mediciones
mediciones_final = df_limpio.groupby('Household_ID').size()
print(f"\nMediciones por casa:")
print(mediciones_final.value_counts().sort_index())

# Verificar que la casa H12857 aún existe con 7 mediciones
if 'H12857' in df_limpio['Household_ID'].values:
    num_med_h12857 = len(df_limpio[df_limpio['Household_ID'] == 'H12857'])
    print(f"\n✓ Casa H12857 mantenida con {num_med_h12857} mediciones")
    print("\nDatos de la casa H12857:")
    print(df_limpio[df_limpio['Household_ID'] == 'H12857'])

# Verificar Has_AC
print(f"\nDistribución de Has_AC:")
print(df_limpio['Has_AC'].value_counts().sort_index())
print(f"  0 (sin AC): {(df_limpio['Has_AC'] == 0).sum()} registros")
print(f"  1 (con AC): {(df_limpio['Has_AC'] == 1).sum()} registros")

# Verificar datos faltantes
print(f"\nDatos faltantes en dataset limpio: {df_limpio.isnull().sum().sum()}")

print("\n" + "=" * 70)
print("PRIMERAS 10 FILAS DEL DATASET LIMPIO")
print("=" * 70)
print(df_limpio.head(10))

# ============================================================================
# 6. GUARDAR DATASET LIMPIO
# ============================================================================
print("\n" + "=" * 70)
print("6. GUARDAR DATASET LIMPIO")
print("=" * 70)

output_path = '/mnt/user-data/outputs/household_energy_consumption_clean.csv'
df_limpio.to_csv(output_path, index=False)
print(f"\n✓ Dataset limpio guardado exitosamente en:")
print(f"  {output_path}")

print("\n" + "=" * 70)
print("RESUMEN DE LA LIMPIEZA")
print("=" * 70)
print(f"✓ Variable Has_AC convertida a binaria (1=Sí, 0=No)")
print(f"✓ Verificación de datos faltantes completada")
print(f"✓ Eliminada solo la fila extra (manteniendo las casas)")
print(f"✓ Dataset final: {df_limpio.shape[0]} filas × {df_limpio.shape[1]} columnas")
print(f"✓ Total de casas: {df_limpio['Household_ID'].nunique()}")
print(f"✓ Todas las casas tienen exactamente 7 mediciones")
print("=" * 70)

Cargando dataset...
ANÁLISIS INICIAL DEL DATASET

Dimensiones originales: (90000, 7)
Columnas: ['Household_ID', 'Date', 'Energy_Consumption_kWh', 'Household_Size', 'Avg_Temperature_C', 'Has_AC', 'Peak_Hours_Usage_kWh']

1. VERIFICACIÓN DE DATOS FALTANTES

Datos faltantes por columna:
Household_ID              0
Date                      0
Energy_Consumption_kWh    0
Household_Size            0
Avg_Temperature_C         0
Has_AC                    0
Peak_Hours_Usage_kWh      0
dtype: int64

Total de filas con datos faltantes: 0
✓ No hay datos faltantes en el dataset

2. ANÁLISIS DE MEDICIONES POR CASA

Estadísticas de mediciones por casa:
count    12857.000000
mean         7.000078
std          0.008819
min          7.000000
25%          7.000000
50%          7.000000
75%          7.000000
max          8.000000
dtype: float64

Distribución de mediciones:
7    12856
8        1
Name: count, dtype: int64

Casas con más de 7 mediciones:
  - H12857: 8 mediciones

3. CONVERTIR HAS_AC A VALORE

Aplicar estadistica Desriptiva

Cargando dataset limpio...
ESTADÍSTICA DESCRIPTIVA - ANÁLISIS COMPLETO

1. INFORMACIÓN GENERAL DEL DATASET

Dimensiones: 89999 filas × 7 columnas
Total de casas únicas: 12857
Rango de fechas: 2025-04-01 a 2025-04-07

Tipos de datos:
Household_ID               object
Date                       object
Energy_Consumption_kWh    float64
Household_Size              int64
Avg_Temperature_C         float64
Has_AC                      int64
Peak_Hours_Usage_kWh      float64
dtype: object

Primeras 5 filas:
  Household_ID        Date  Energy_Consumption_kWh  Household_Size  \
0       H00001  2025-04-01                     8.4               4   
1       H00001  2025-04-02                     7.9               4   
2       H00001  2025-04-03                     9.2               4   
3       H00001  2025-04-04                     7.9               4   
4       H00001  2025-04-05                     9.6               4   

   Avg_Temperature_C  Has_AC  Peak_Hours_Usage_kWh  
0               17.8  